
# Comment section of the excersice

Ball tracking with a Kalman filter

Detection. Each frame is converted to HSV and thresholded on the ball’s colour, which gives a binary mask. 
We clean the mask with an opening and a closing, keep the largest contour, and use minEnclosingCircle to get the ball’s centre. 
That centre is the measurement for the filter. If no blob is large enough, we treat the ball as not visible.

Kalman filter. The state has six variables: position, velocity and acceleration in x and y. 
F comes from the constant-acceleration equations with dt = 1 (p' = p + v + ½a, v' = v + a, a' = a), 
with one block per axis because x and y are independent. H selects only the positions, since that is all the camera measures. 
P starts large because we know nothing about the initial state, and R sets how much we trust the detector.

Tracking and occlusion. In every frame we run the update only if the ball was detected, and the prediction always. 
When the ball is hidden, the filter keeps moving the estimate using the last velocity and acceleration it inferred, while P grows. 
This lets it follow the ball behind the obstacle, and when the ball reappears it corrects itself within a few frames.


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output 

VIDEO_IN  = "rolling_ball.mp4"            # input video (use "rolling_ball_challenge.mp4" for the challenge)


def update(x, P, Z, H, R):
    y = Z - np.dot(H, x)                          # Innovation
    S = np.dot(H, np.dot(P, H.T)) + R             # Innovation covariance
    K = np.dot(P, np.dot(H.T, np.linalg.pinv(S))) # Kalman gain
    x = x + np.dot(K, y)                          # State update
    P = np.dot((np.eye(6) - np.dot(K, H)), P)           # Covariance update
    return [x, P]

    ### Insert update function
   
    
def predict(x, P, F, u):
    x = np.dot(F, x) + u          # Predict next state
    P = np.dot(F, np.dot(P, F.T)) # Predict next covariance
    return [x, P]
    ### insert predict function
        
### Initialize Kalman filter ###
dt = 1.
# The initial state (6x1).
x = np.zeros((6, 1))

# The initial uncertainty (6x6).
P = 1000. * np.eye(6)
# The external motion (6x1).
u = np.zeros((6, 1))

# The transition matrix (6x6). 
F = np.array([[1, dt, 0.5*dt**2, 0, 0,  0        ],
              [0, 1,  dt,        0, 0,  0        ],
              [0, 0,  1,         0, 0,  0        ],
              [0, 0,  0,         1, dt, 0.5*dt**2],
              [0, 0,  0,         0, 1,  dt       ],
              [0, 0,  0,         0, 0,  1        ]])

# The observation matrix (2x6).
H = np.array([[1, 0, 0, 0, 0, 0],
              [0, 0, 0, 1, 0, 0]])

# The measurement uncertainty.
R = np.eye(2) * 1.


#Detect ball by color
LOWER = np.array([0, 100, 100]) # for detecting red in the picture
UPPER = np.array([10, 255, 255])
MIN_AREA = 500 

def detect_ball(frame, lower=LOWER, upper=UPPER,min_area=MIN_AREA):
    hsv=cv2.cvtColor(frame,cv2.COLOR_BGR2HSV)
    mask= cv2.inRange(hsv,lower,upper)
    # Clean the mask: remove small specks, then fill small holes
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, mask

    c = max(contours, key=cv2.contourArea)                   # largest blob
    if cv2.contourArea(c) < min_area:
        return None, mask

    (cx, cy), radius = cv2.minEnclosingCircle(c)
    return (cx, cy, radius), mask


# Load the video

cap = cv2.VideoCapture(VIDEO_IN) 
radius2=0;
# Looping through all the frames
while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    ### Detect the ball ###
    ball,mask=detect_ball(frame)

    ### If the ball is found, update the Kalman filter ###
    if ball is not None:
        cx,cy,radius=ball
        radius2=radius
        Z=np.array([[cx],[cy]])
        x,P=update(x,P,Z,H,R)
        cv2.circle(frame, (int(cx), int(cy)), int(radius2), (0, 0, 255), 3) 
    ### Predict the next state
    x, P = predict(x, P, F, u)
    cv2.circle(frame, (int(x[0, 0]), int(x[3, 0])),int(radius2), (255, 0, 0), 3)
    
    ### Draw the current tracked state and the predicted state on the image frame ###
    #cv2.circle(frame, (int(x[0, 0]), int(x[3, 0])), 10, (0, 255, 0), -1)
    # Show the frame
    cv2.imshow('Frame', frame)
    cv2.waitKey(50)
    
    small = cv2.resize(frame, (960, 540))             # smaller image = faster display
    clear_output(wait=True)                           # replace the previous frame
    plt.figure(figsize=(10, 5.6))
    plt.imshow(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))   # OpenCV uses BGR, matplotlib uses RGB
    plt.title("red = detection, green = current state, blue = prediction")
    plt.title(f"pos=({x[0,0]:.0f}, {x[3,0]:.0f})  "
    f"vel=({x[1,0]:.1f}, {x[4,0]:.1f})  "
    f"acc=({x[2,0]:.2f}, {x[5,0]:.2f})")
    plt.axis("off")
    plt.show()
    

cap.release()
